# ARCHS4 CLAMP models with BP prior at 100% sample coverage (Random Sampling)

**Environment:** `clamp-analyses`  

This notebook builds 3 CLAMP models using the BP (GO Biological Process) prior at 100% sample coverage, each with a different random seed for the randomized SVD.

Since this is 100% coverage, no sample subselection is performed — all samples are used in every run.

Steps (repeated for each seed):
1. Compute SVD on the full preprocessed ARCHS4 data
2. Run CLAMPbase
3. Run CLAMPfull with BP prior

Each run uses a different seed (`base_seed + 0:2`) for reproducibility, allowing comparison of model stability across different SVD randomizations.

## Load libraries

In [1]:
if (!requireNamespace("CLAMP", quietly = TRUE)) {
    REPO_PATH <- "/home/msubirana/Documents/pivlab/CLAMP" 
    remotes::install_local(REPO_PATH, force = TRUE, dependencies = FALSE)
}

library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(CLAMP)
library(rhdf5)

source(here("config.R"))


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Matrix

Loaded glmnet 4.1-10

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



## Configuration

In [ ]:
# Base output directory
base_output_dir <- config$ARCHS4$DATASET_FOLDER

# Coverage level for this notebook
coverage <- 1.0
coverage_pct <- coverage * 100

# Data path for priors
data_path <- here::here('data/archs4')
archs4_file <- here('data/archs4/human_gene_v2.5.h5')

# SVD parameters
N_CORES <- config$ARCHS4$CLAMP_PARAMS$RANDOM_SVD_N_CORES

# Seeds for reproducibility - run 3 models with different seeds
base_seed <- config$ARCHS4$CLAMP_PARAMS$RANDOM_SVD_SEED
seeds <- base_seed + 0:2
n_runs <- length(seeds)
message("Will run ", n_runs, " models with seeds: ", paste(seeds, collapse = ", "))

## Load preprocessed data

In [3]:
# Load metadata
meta <- readRDS(file.path(base_output_dir, "metadata_filtered.rds"))
n_genes_thin <- meta$n_genes_thin
n_samples_total <- meta$n_samples
archs4_genes <- meta$gene_symbols_thin

# Load sample names
all_samples <- readRDS(file.path(base_output_dir, "all_samples.rds"))
sample_names_total <- all_samples[seq_len(n_samples_total)]

# Load BP pathway
BP_pathMat <- readRDS(file.path(data_path, "BP_pathMat.rds"))
BP_matched <- getMatchedPathwayMat(BP_pathMat, archs4_genes)
message("Loaded and matched BP pathway matrix")

# Load FBM
fbm_file <- file.path(base_output_dir, "fbm")
output_file <- paste0(fbm_file, "_filtered")

archs4_fbm_filt <- FBM(
  nrow        = n_genes_thin,
  ncol        = n_samples_total,
  backingfile = output_file,
  create_bk   = FALSE
)

message("Loaded FBM with ", n_genes_thin, " genes and ", n_samples_total, " samples")

There are 11936 genes in the intersection between data and prior

Removing 2059 pathways

Loaded and matched BP pathway matrix

Loaded FBM with 18423 genes and 605614 samples



## Run 3 CLAMP models with different seeds (all samples)

In [ ]:
n_genes <- nrow(archs4_fbm_filt)
n_samples <- n_samples_total
sample_names <- sample_names_total

message("Using all ", n_samples, " samples (100% coverage)")

# Store results summary
results_summary <- data.frame(
  run = integer(),
  seed = integer(),
  n_samples = integer(),
  CLAMP_K = integer(),
  stringsAsFactors = FALSE
)

for (run_idx in seq_len(n_runs)) {
  current_seed <- seeds[run_idx]
  message("\n", strrep("=", 60))
  message("RUN ", run_idx, "/", n_runs, " - Seed: ", current_seed)
  message(strrep("=", 60))
  
  # Create output directory for this run
  output_dir <- file.path(base_output_dir, paste0("bp_coverage_rs", coverage_pct, "_seed_", run_idx))
  dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)
  
  # No subsampling - use the full FBM directly
  Y <- archs4_fbm_filt
  
  # SVD with current seed
  message("Computing SVD with seed ", current_seed, "...")
  SVD_K <- min(n_samples - 1, n_genes - 1)
  
  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = FALSE)
    blas_nproc <- getOption("default.nproc.blas")
    options(default.nproc.blas = NULL)
  }
  
  set.seed(current_seed)
  svd_result <- big_randomSVD(Y, k = SVD_K, ncores = N_CORES)
  
  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = TRUE)
    options(default.nproc.blas = blas_nproc)
  }
  
  valid_idx <- which(!is.nan(svd_result$d))
  svd_result$d <- svd_result$d[valid_idx]
  svd_result$u <- svd_result$u[, valid_idx, drop = FALSE]
  svd_result$v <- svd_result$v[, valid_idx, drop = FALSE]
  
  saveRDS(svd_result, file = file.path(output_dir, "svd.rds"))
  
  # Estimate CLAMP K
  svd_list <- list(d = svd_result$d)
  CLAMP_K <- num.pc(svd_list) * 2
  message("CLAMP K = ", CLAMP_K)
  saveRDS(CLAMP_K, file = file.path(output_dir, "CLAMP_K.rds"))
  
  # CLAMPbase
  message("Running CLAMPbase...")
  baseRes <- CLAMPbase(
    Y = Y,
    svdres = svd_result,
    trace = TRUE,
    clamp_k = CLAMP_K
  )
  
  baseRes$Z <- data.frame(baseRes$Z)
  rownames(baseRes$Z) <- archs4_genes
  baseRes$B <- data.frame(baseRes$B)
  colnames(baseRes$B) <- sample_names
  
  saveRDS(baseRes, file = file.path(output_dir, "CLAMPbase.rds"))
  
  model_dir <- file.path(output_dir, "CLAMPbase")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  write.csv(baseRes$B, file.path(model_dir, "B.csv"))
  write.csv(baseRes$Z, file.path(model_dir, "Z.csv"))
  
  # CLAMPfull with BP prior
  message("Running CLAMPfull with BP prior...")
  fullRes <- CLAMPfull(
    Y = Y,
    svdres = svd_result,
    priorMat = BP_matched,
    clamp.base.result = baseRes,
    use_cpp = TRUE,
    trace = TRUE,
    clamp_k = CLAMP_K
  )
  
  fullRes$Z <- data.frame(fullRes$Z)
  rownames(fullRes$Z) <- archs4_genes
  fullRes$B <- data.frame(fullRes$B)
  colnames(fullRes$B) <- sample_names
  fullRes$summary <- fullRes$summary %>%
    dplyr::rename(LV = LV_index) %>%
    dplyr::mutate(LV = paste0('LV', LV))
  
  saveRDS(fullRes, file = file.path(output_dir, "CLAMPfull_BP.rds"))
  
  model_dir <- file.path(output_dir, "CLAMPfull_BP")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  write.csv(fullRes$B, file.path(model_dir, "B.csv"))
  write.csv(fullRes$Z, file.path(model_dir, "Z.csv"))
  write.csv(fullRes$summary, file.path(model_dir, "summary.csv"))
  
  # Store summary
  results_summary <- rbind(results_summary, data.frame(
    run = run_idx,
    seed = current_seed,
    n_samples = n_samples,
    CLAMP_K = CLAMP_K
  ))
  
  # Clean up memory
  rm(svd_result, baseRes, fullRes)
  gc()
}

message("\n", strrep("=", 60))
message("All ", n_runs, " runs completed!")
message(strrep("=", 60))